# 04. Model Training, Validation & Evaluation

**Objective:** Train the 1D-CNN Market Signal Encoder using the Adam optimizer and Binary Cross-Entropy loss with early stopping, evaluate on the untouched test partition, and document comparative performance against published metrics.

---

## 1. Experimental Configuration
- **Optimizer:** Adam (Learning Rate $\alpha = 0.001$, Weight Decay $= 10^{-4}$)
- **Loss Function:** Binary Cross-Entropy with Logits (`nn.BCEWithLogitsLoss`)
- **Batch Size:** $64$
- **Early Stopping Patience:** $7$ epochs
- **Random Seed:** $42$


In [1]:
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append("..")
from src import config, evaluation, visualization

print("Evaluation suite loaded.")


Evaluation suite loaded.


## 2. Evaluation Metrics on Holdout Test Set
We compute Accuracy, Precision, Recall, F1-Score, ROC-AUC, and the full Confusion Matrix on untouched test data.


In [2]:
metrics_path = Path("../Results/evaluation_metrics.csv")
if metrics_path.exists():
    df_metrics = pd.read_csv(metrics_path)
    print("Test Set Performance Summary:")
    print(df_metrics.to_string(index=False))
else:
    print("Metrics file not yet generated. Complete training run to populate.")


Test Set Performance Summary:
                 Metric        Value                                          Notes
               Accuracy     0.824641                            Overall correctness
    Precision (Class 1)     0.793274 Precision for Entry level (Low Priority proxy)
       Recall (Class 1)     0.742269    Recall for Entry level (Low Priority proxy)
     F1-Score (Class 1)     0.766925            Harmonic mean of precision & recall
                ROC-AUC     0.893290                           Area under ROC curve
                 PR-AUC     0.849567              Area under Precision-Recall curve
    True Positives (TP)  5449.000000               Correctly classified Entry level
   False Positives (FP)  1420.000000            Higher level misclassified as Entry
    True Negatives (TN) 10126.000000                 Correctly classified Non-entry
   False Negatives (FN)  1892.000000      Entry level misclassified as Higher level
Total Test Observations 18887.000000          

## 3. Identification of Published Inconsistency
In Section 4 of the published paper, the authors state:
- Reported Test Size: $N = 24,823$
- Published Confusion Matrix:
  - $TN = 4,821$
  - $FP = 1,012$
  - $FN = 734$
  - $TP = 3,363$
  - **Sum of Matrix Entries:** $4,821 + 1,012 + 734 + 3,363 = 9,930$

This reflects a **2.5x mismatch** in the published literature. For research integrity, we document this discrepancy and do not fabricate data to fit either number.


In [3]:
pub_matrix_sum = 4821 + 1012 + 734 + 3363
print(f"Published Claimed Test Size: {config.PUBLISHED_REPORTED_TEST_N:,}")
print(f"Published Confusion Matrix Sum: {pub_matrix_sum:,}")
print(f"Discrepancy Factor: {config.PUBLISHED_REPORTED_TEST_N / pub_matrix_sum:.2f}x")


Published Claimed Test Size: 24,823
Published Confusion Matrix Sum: 9,930
Discrepancy Factor: 2.50x
